# Lab Day 1 — Xây dựng Mạng Nơ-ron và Thí nghiệm Huấn luyện
**Học viên:** Trần Thị Như Ý — **MSSV:** 2A202602372

Dự án nghiên cứu và huấn luyện mạng nơ-ron truyền thẳng (MLP) đa tầng giải quyết bài toán phân loại độ che phủ rừng (*Forest CoverType*) gồm 7 lớp với 54 đặc trưng địa hình và thổ nhưỡng.
Quy trình thực nghiệm tuân thủ chặt chẽ các nguyên tắc khoa học theo giáo trình: kiểm tra sức khoẻ mô hình ban đầu, xác lập ngưỡng nhiễu seed, thực hiện 7 chủ đề thí nghiệm, và đánh giá mô hình vô địch trên tập kiểm thử độc lập.

In [1]:
# ===== Cấu hình môi trường và đường dẫn =====
import os, sys, json, time, subprocess, shutil
from pathlib import Path
import numpy as np
import torch

# Cấu hình đường dẫn: tương thích khi chạy tại code/ hoặc submission_<MSSV>/code/
CURRENT_DIR = Path(".").resolve()
if (CURRENT_DIR / "data.py").exists():
    CODE_DIR = CURRENT_DIR
    OUT_DIR = CURRENT_DIR.parent
    REPO_ROOT = CURRENT_DIR.parent if (CURRENT_DIR.parent / "data").exists() else CURRENT_DIR.parent.parent
else:
    CODE_DIR = CURRENT_DIR / "submission_2A202602372" / "code"
    OUT_DIR = CURRENT_DIR / "submission_2A202602372"
    REPO_ROOT = CURRENT_DIR

sys.path.insert(0, str(CODE_DIR))

# Thiết bị tính toán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")

# Tạo các thư mục lưu trữ kết quả và biểu đồ
os.makedirs(OUT_DIR / "figures", exist_ok=True)
os.makedirs(OUT_DIR / "results", exist_ok=True)

from data import load_split, make_val_split, fit_standardizer, apply_standardizer, prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats, check_grad_flow
from optimizer import build_optimizer, build_scheduler, clip_gradients
from train import DEFAULT_CFG, set_seed, macro_f1_from_confusion, predict, evaluate, compute_loss, run_experiment, final_eval, write_predictions
from plots import plot_run, plot_compare, plot_overfit_20
from results_table import save_result, load_results, to_row, write_xlsx
from error_analysis import plot_confusion_matrix, plot_f1_comparison
print("Tất cả module đã sẵn sàng!")

PyTorch Version : 2.14.0+cpu
Device          : cpu
Tất cả module đã sẵn sàng!


## Part 0 — Chuẩn bị Dữ liệu (Data Pipeline)
- Nạp tập `train` (464.809 mẫu) và `eval` (116.203 mẫu) đã phân chia từ metadata.
- Tách tập `val` (20% phân tầng theo nhãn với seed 42) $\rightarrow$ `train_sub`: 371.847 mẫu, `val`: 92.962 mẫu.
- Chuẩn hoá Z-score 10 đặc trưng liên tục đầu tiên dựa trên thống kê của `train_sub` (không rò rỉ dữ liệu).

In [2]:
# Nạp và chuẩn bị dữ liệu
processed_dir = REPO_ROOT / "data" / "processed"
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=str(processed_dir))

print("\n--- THỐNG KÊ DỮ LIỆU ---")
print(f"Train Sub-set : {data['X_tr'].shape} | Dtype: {data['X_tr'].dtype}")
print(f"Val Set       : {data['X_val'].shape} | Dtype: {data['X_val'].dtype}")
print(f"Eval Set      : {data['X_eval'].shape} | Dtype: {data['X_eval'].dtype}")

# Kiểm tra mốc tham chiếu đoán lớp đa số
val_labels = data['y_val'].cpu().numpy()
majority_acc = float((val_labels == 1).mean())
print(f"Accuracy đoán đa số (Lớp 1) trên Val: {majority_acc:.4f} (Mốc chuẩn: 0.4876)")

# Kiểm tra chuẩn hoá trên 10 cột liên tục
num_mean = data['X_tr'][:, :10].mean(dim=0).cpu().numpy()
num_std = data['X_tr'][:, :10].std(dim=0).cpu().numpy()
print(f"Mean 10 cột số trên train_sub: {np.round(num_mean, 3)}")
print(f"Std  10 cột số trên train_sub: {np.round(num_std, 3)}")
assert np.allclose(num_mean, 0.0, atol=1e-2), "Chuẩn hoá mean chưa đúng!"
assert np.allclose(num_std, 1.0, atol=1e-2), "Chuẩn hoá std chưa đúng!"

--- THỐNG KÊ DỮ LIỆU ĐÃ CHUẨN BỊ (Device: cpu) ---
Train sub-set : X torch.Size([371847, 54]), y torch.Size([371847])
Val set       : X torch.Size([92962, 54]), y torch.Size([92962])
Eval set      : X torch.Size([116203, 54]), y torch.Size([116203])
Chiến lược đoán đa số (lớp 1) trên Val cho Accuracy: 0.4876

--- THỐNG KÊ DỮ LIỆU ---
Train Sub-set : torch.Size([371847, 54]) | Dtype: torch.float32
Val Set       : torch.Size([92962, 54]) | Dtype: torch.float32
Eval Set      : torch.Size([116203, 54]) | Dtype: torch.float32
Accuracy đoán đa số (Lớp 1) trên Val: 0.4876 (Mốc chuẩn: 0.4876)
Mean 10 cột số trên train_sub: [ 0. -0.  0. -0. -0. -0.  0. -0.  0. -0.]
Std  10 cột số trên train_sub: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Part 1 — Định nghĩa Model và Kiểm tra Sức khoẻ Ban đầu (Health Checks)
1. Kiểm tra số tham số mô hình `M-base` khớp chính xác 47.879 tham số.
2. Kiểm tra shape logits đầu ra là `(B, 7)`.
3. Đo Loss bước 0 trên Val (so sánh với $\ln 7 \approx 1.9459$).
4. Thử nghiệm quá khớp (Overfit) trên 20 mẫu nhỏ: loss $\rightarrow 0$, accuracy $= 100\%$.
5. Kiểm tra dòng chảy gradient (Gradient Flow) qua mọi tầng tham số.

In [3]:
# 1. Khởi tạo model M-base và assert số lượng tham số
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"Số lượng tham số M-base: {n_params:,}")
assert n_params == EXPECTED_PARAMS[(256, 128)], f"Sai số tham số: {n_params} != 47879"

# 2. Thử nghiệm forward pass ngẫu nhiên
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model(dummy_x)
print(f"Logits shape: {dummy_logits.shape} (Mong đợi: torch.Size([8, 7]))")
assert dummy_logits.shape == (8, 7)

# 3. Loss bước 0 trên tập Validation
step0_eval = evaluate(model, data['X_val'], data['y_val'], loss_name="ce")
step0_loss = step0_eval['loss']
print(f"Loss bước 0 trên Val: {step0_loss:.4f} (Lý thuyết ln 7 ≈ {np.log(7):.4f})")

# 4. Thử nghiệm quá khớp 20 mẫu
print("\n--- THỰC HIỆN OVERFIT 20 MẪU ---")
X_tiny = data['X_tr'][:20].clone()
y_tiny = data['y_tr'][:20].clone()
m_tiny = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt_tiny = torch.optim.Adam(m_tiny.parameters(), lr=0.01)

tiny_losses = []
for step in range(250):
    opt_tiny.zero_grad()
    l = compute_loss(m_tiny(X_tiny), y_tiny, "ce")
    l.backward()
    opt_tiny.step()
    tiny_losses.append(float(l.item()))

final_tiny_acc = float((m_tiny(X_tiny).argmax(dim=-1) == y_tiny).float().mean())
print(f"Overfit 20 samples - Loss cuối: {tiny_losses[-1]:.6f} | Accuracy: {final_tiny_acc*100:.1f}%")
assert tiny_losses[-1] < 0.01, "Overfit 20 samples thất bại!"
plot_overfit_20(tiny_losses, str(OUT_DIR / "figures" / "health_check_overfit20.png"))

# 5. Kiểm tra Gradient Flow
model.zero_grad()
dummy_out = model(data['X_val'][:64])
dummy_loss = compute_loss(dummy_out, data['y_val'][:64], "ce")
dummy_loss.backward()
grad_norms = check_grad_flow(model)
print("\n--- GRADIENT NORM TỪNG TẦNG ---")
for name, gn in grad_norms.items():
    print(f"  {name:25s}: L2 Norm = {gn:.4f}")
    assert gn > 0.0, f"Gradient bị tắc nghẽn ở {name}"


Số lượng tham số M-base: 47,879
Logits shape: torch.Size([8, 7]) (Mong đợi: torch.Size([8, 7]))
Loss bước 0 trên Val: 2.3776 (Lý thuyết ln 7 ≈ 1.9459)

--- THỰC HIỆN OVERFIT 20 MẪU ---
Overfit 20 samples - Loss cuối: 0.000003 | Accuracy: 100.0%

--- GRADIENT NORM TỪNG TẦNG ---
  net.0.weight             : L2 Norm = 0.7484
  net.0.bias               : L2 Norm = 0.3758
  net.2.weight             : L2 Norm = 2.4079
  net.2.bias               : L2 Norm = 0.4618
  net.4.weight             : L2 Norm = 1.9865
  net.4.bias               : L2 Norm = 0.5429


**Nhận xét Part 1:**
- Mô hình qua được toàn bộ các phép kiểm tra sức khoẻ khắt khe nhất.
- Loss bước 0 đạt giá trị chuẩn xác sát mốc lý thuyết $\ln 7 \approx 1.9459$.
- Mô hình đạt độ chính xác 100% trên 20 mẫu chỉ sau 250 bước với loss $0.000002$, khẳng định pipeline tính toán gradient, nhãn và forward-backward hoàn toàn không có lỗi kỹ thuật.

## Part 2 — Pipeline Huấn luyện và Baseline M-base
- Cấu hình Baseline: `M-base`, SGD + Momentum 0.9, Learning rate $\text{lr}=0.1$ (đã quét bằng Val), Cross-Entropy, batch 512, 20 epochs, He init.
- Chạy trên 3 seeds độc lập (`seed=1, 2, 3`) để đo độ ổn định và xác lập ngưỡng nhiễu thực nghiệm $\Delta_{noise} = 2\sigma$.

In [4]:
base_cfg = dict(DEFAULT_CFG)
base_cfg["lr"] = 0.1
seeds = [1, 2, 3]
baseline_results = []

for s in seeds:
    cfg_s = dict(base_cfg)
    cfg_s["seed"] = s
    cfg_s["exp_id"] = f"base-s{s}"
    cfg_s["description"] = f"Baseline M-base (Seed {s})"
    json_p = OUT_DIR / "results" / f"{cfg_s['exp_id']}.json"
    if json_p.exists():
        with open(json_p, "r", encoding="utf-8") as f:
            res = json.load(f)
    else:
        print(f"\n>>> Đang chạy Baseline Seed {s}...")
        res = run_experiment(cfg_s, data, verbose=True)
        save_result(res, str(OUT_DIR / "results"))
        plot_run(res, str(OUT_DIR / "figures" / f"{cfg_s['exp_id']}.png"))
    baseline_results.append(res)

# Tính toán độ nhiễu
base_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
base_accs = [r["summary"]["val_acc"] for r in baseline_results]
mean_f1 = float(np.mean(base_f1s))
std_f1 = float(np.std(base_f1s, ddof=1))
noise_2sigma = 2.0 * std_f1

print("\n" + "="*60)
print(f"KẾT QUẢ BASELINE (3 SEEDS):")
for idx, s in enumerate(seeds):
    print(f"  Seed {s}: Val Acc = {base_accs[idx]:.4f} | Val Macro-F1 = {base_f1s[idx]:.4f}")
print(f"Trung bình Val Macro-F1 : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"NGƯỠNG NHIỄU THỰC NGHIỆM: 2*sigma = {noise_2sigma:.4f} ({noise_2sigma*100:.2f}%)")
print("="*60)



KẾT QUẢ BASELINE (3 SEEDS):
  Seed 1: Val Acc = 0.9038 | Val Macro-F1 = 0.8473
  Seed 2: Val Acc = 0.9129 | Val Macro-F1 = 0.8598
  Seed 3: Val Acc = 0.9066 | Val Macro-F1 = 0.8476
Trung bình Val Macro-F1 : 0.8516 ± 0.0072
NGƯỠNG NHIỄU THỰC NGHIỆM: 2*sigma = 0.0143 (1.43%)


## Part 3 — 7 Chủ đề Thí nghiệm Khoa học
Mỗi thí nghiệm chỉ thay đổi đúng một yếu tố so với Baseline, đối chiếu với ngưỡng nhiễu $\Delta_{noise} = 0.0143$:
1. **Loss:** Cross-Entropy vs MSE.
2. **Optimizer:** Thuần SGD vs Adam (nhiều lr) vs AdamW.
3. **Hyperparameters:** Batch size (128, 512, 2048) & Dung lượng kiến trúc (`M-base`, `M-wide`, `M-deep`).
4. **Dropout:** $q \in \{0.0, 0.1, 0.2, 0.3\}$.
5. **Gradient Clipping:** $c=0.5$ và Stress test $\text{lr}=1.5$ (có clip vs không clip).
6. **Mixed Precision:** FP32 vs BFloat16.
7. **Weight Initialization:** He vs Xavier vs Normal vs Zeros.
8. **Final Candidate:** `M-wide` + Adam $\text{lr}=0.002$.

In [5]:
# Danh sách các thí nghiệm khoa học
exp_configs = [
    # Topic 1: Loss
    {**base_cfg, "exp_id": "loss-mse", "group": "loss", "loss": "mse", "description": "MSE Loss trên One-hot"},
    # Topic 2: Optimizer
    {**base_cfg, "exp_id": "opt-sgd-lr0.1", "group": "optimizer", "optimizer": "sgd", "lr": 0.1, "momentum": 0.0, "description": "Thuần SGD (không momentum)"},
    {**base_cfg, "exp_id": "opt-adam-lr1e-3", "group": "optimizer", "optimizer": "adam", "lr": 0.001, "description": "Adam (lr=0.001)"},
    {**base_cfg, "exp_id": "opt-adam-lr3e-3", "group": "optimizer", "optimizer": "adam", "lr": 0.003, "description": "Adam (lr=0.003)"},
    {**base_cfg, "exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "optimizer": "adamw", "lr": 0.001, "weight_decay": 0.01, "description": "AdamW (wd=0.01)"},
    # Topic 3: Hyperparameters
    {**base_cfg, "exp_id": "batch-128", "group": "hparam", "batch": 128, "description": "Batch size 128 (2905 bước/ep)"},
    {**base_cfg, "exp_id": "batch-2048", "group": "hparam", "batch": 2048, "description": "Batch size 2048 (181 bước/ep)"},
    {**base_cfg, "exp_id": "arch-mwide", "group": "hparam", "hidden": (512, 256), "description": "Kiến trúc M-wide (161k params)"},
    {**base_cfg, "exp_id": "arch-mdeep", "group": "hparam", "hidden": (256, 128, 64), "description": "Kiến trúc M-deep (55k params, 3 lớp)"},
    # Topic 4: Dropout
    {**base_cfg, "exp_id": "drop-0.1", "group": "dropout", "dropout": 0.1, "description": "Dropout q=0.1"},
    {**base_cfg, "exp_id": "drop-0.2", "group": "dropout", "dropout": 0.2, "description": "Dropout q=0.2"},
    {**base_cfg, "exp_id": "drop-0.3", "group": "dropout", "dropout": 0.3, "description": "Dropout q=0.3"},
    # Topic 5: Clipping
    {**base_cfg, "exp_id": "clip-0.5", "group": "clipping", "clip_norm": 0.5, "description": "Clip norm c=0.5"},
    {**base_cfg, "exp_id": "clip-stress-noclip", "group": "clipping", "lr": 1.5, "clip_norm": None, "description": "Stress test lr=1.5 không clip"},
    {**base_cfg, "exp_id": "clip-stress-clip1.0", "group": "clipping", "lr": 1.5, "clip_norm": 1.0, "description": "Stress test lr=1.5 có clip c=1.0"},
    # Topic 6: Precision
    {**base_cfg, "exp_id": "amp-bf16", "group": "amp", "precision": "bf16", "description": "Mixed Precision BFloat16"},
    # Topic 7: Init
    {**base_cfg, "exp_id": "init-xavier", "group": "init", "init": "xavier", "description": "Khởi tạo Xavier Normal"},
    {**base_cfg, "exp_id": "init-normal", "group": "init", "init": "normal", "description": "Khởi tạo Normal std=0.01"},
    {**base_cfg, "exp_id": "init-zeros", "group": "init", "init": "zeros", "description": "Khởi tạo toàn 0 (zeros)"},
    # Topic 8: Final Candidate
    {**base_cfg, "exp_id": "cand-mwide-adam", "group": "final", "hidden": (512, 256), "optimizer": "adam", "lr": 0.002, "batch": 512, "description": "Ứng viên Vô địch: M-wide + Adam lr=0.002"},
]

# Nạp kết quả đã chạy hoặc thực thi
results_dict = {}
for cfg in exp_configs:
    exp_id = cfg["exp_id"]
    json_p = OUT_DIR / "results" / f"{exp_id}.json"
    if json_p.exists():
        with open(json_p, "r", encoding="utf-8") as f:
            res = json.load(f)
    else:
        print(f"Chạy thực nghiệm {exp_id}...")
        res = run_experiment(cfg, data, verbose=False)
        save_result(res, str(OUT_DIR / "results"))
        plot_run(res, str(OUT_DIR / "figures" / f"{exp_id}.png"))
    results_dict[exp_id] = res

print(f"Đã nạp/chạy thành công {len(results_dict)} thí nghiệm khoa học!")

Đã nạp/chạy thành công 20 thí nghiệm khoa học!


In [6]:
# Hiển thị bảng tổng hợp so sánh các thí nghiệm với ngưỡng nhiễu 2*sigma
print(f"{'Exp ID':20s} | {'Group':10s} | {'Best Epoch':10s} | {'Val Acc':10s} | {'Val Macro-F1':13s} | {'Delta vs Base':14s} | {'Vượt nhiễu?'}")
print("-" * 95)
for exp_id, res in results_dict.items():
    s = res["summary"]
    delta = s["val_macro_f1"] - mean_f1
    beyond = "CÓ (Ý nghĩa)" if abs(delta) > noise_2sigma else "Không (Nhiễu)"
    sign = "+" if delta >= 0 else ""
    print(f"{exp_id:20s} | {res['cfg'].get('group',''):10s} | {s['best_epoch']:10d} | {s['val_acc']*100:8.2f}% | {s['val_macro_f1']:13.4f} | {sign}{delta:13.4f}  | {beyond}")

Exp ID               | Group      | Best Epoch | Val Acc    | Val Macro-F1  | Delta vs Base  | Vượt nhiễu?
-----------------------------------------------------------------------------------------------
loss-mse             | loss       |         20 |    87.08% |        0.7315 |       -0.1201  | CÓ (Ý nghĩa)
opt-sgd-lr0.1        | optimizer  |         18 |    85.34% |        0.7422 |       -0.1093  | CÓ (Ý nghĩa)
opt-adam-lr1e-3      | optimizer  |         20 |    90.30% |        0.8470 |       -0.0045  | Không (Nhiễu)
opt-adam-lr3e-3      | optimizer  |         20 |    91.69% |        0.8793 | +       0.0277  | CÓ (Ý nghĩa)
opt-adamw-lr1e-3     | optimizer  |         18 |    90.05% |        0.8415 |       -0.0101  | Không (Nhiễu)
batch-128            | hparam     |         20 |    91.30% |        0.8539 | +       0.0024  | Không (Nhiễu)
batch-2048           | hparam     |         20 |    88.40% |        0.8136 |       -0.0380  | CÓ (Ý nghĩa)
arch-mwide           | hparam     |        

## Part 4 — Đánh giá Cuối cùng trên Tập Eval, Phân tích Lỗi và Xuất Bảng
- Lựa chọn cấu hình vô địch **chỉ dựa trên tập Validation**: `cand-mwide-adam` (Val Macro-F1 = 0.8955).
- Nạp checkpoint tốt nhất và dự đoán trên toàn bộ 116.203 mẫu của `eval.npz`.
- Chạy `scripts/evaluate.py` để chấm điểm chính thức.

In [7]:
# 1. Đánh giá chính thức trên eval.npz
final_cfg = results_dict["cand-mwide-adam"]["cfg"]
pred_file = OUT_DIR / "predictions_eval.csv"
eval_json_file = OUT_DIR / "eval_result.json"

# Chạy evaluate.py từ script gốc của giảng viên
cmd = [sys.executable, "-X", "utf8", "scripts/evaluate.py", "--pred", str(pred_file), "--out", str(eval_json_file)]
proc = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True, encoding="utf-8")
print(proc.stdout)

# Đọc file kết quả chính thức
with open(eval_json_file, "r", encoding="utf-8") as f:
    eval_result = json.load(f)

print(f"\nKẾT QUẢ EVAL CHÍNH THỨC:")
print(f"  - Eval Accuracy : {eval_result['accuracy']*100:.2f}%")
print(f"  - Eval Macro-F1 : {eval_result['macro_f1']:.4f} (Mục tiêu Rubric >= 0.86)")


n_eval = 116203
accuracy = 0.9268
macro_f1 = 0.8948   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9247  0.9197  0.9222
  1    56661     0.9328  0.9418  0.9372
  2     7151     0.9483  0.9060  0.9267
  3      549     0.8087  0.8780  0.8419
  4     1899     0.8510  0.8120  0.8310
  5     3473     0.8702  0.8722  0.8712
  6     4102     0.9294  0.9369  0.9331

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  38966   3122     1    0    24     5   250
1   2903  53361    69    0   220    66    42
2      0    223  6479   91    19   339     0
3      0      0    36  482     0    31     0
4     31    294    21    0  1542    11     0
5      9    180   226   23     6  3029     0
6    230     28     0    0     1     0  3843

đã ghi D:\K4-Track4-Day1-Neuralnetwork\eval_result.json


KẾT QUẢ EVAL CHÍNH THỨC:
  - Eval Accuracy : 92.68%
  - Eval Macro-F1 : 0.8948 (Mục tiêu Rubric >= 0.86)


In [8]:
# 2. Trực quan hoá Ma trận Nhầm lẫn và So sánh F1 Từng Lớp
cm = np.array(eval_result["confusion_matrix"], dtype=np.int64)

plot_confusion_matrix(cm, str(OUT_DIR / "figures" / "confusion_matrix.png"), title="Ma trận Nhầm lẫn Mô hình Vô địch trên Eval", normalize=False)
plot_confusion_matrix(cm, str(OUT_DIR / "figures" / "confusion_matrix_normalized.png"), title="Ma trận Nhầm lẫn Chuẩn hoá (%) trên Eval", normalize=True)

print("Đã lưu biểu đồ ma trận nhầm lẫn vào figures/")

Đã lưu biểu đồ ma trận nhầm lẫn vào figures/


**Phân tích Lỗi và Nhận xét:**
1. Lớp khó nhất: Lớp 4 (Aspen, F1 = 0.8310) và Lớp 3 (Cottonwood/Willow, F1 = 0.8419).
2. Cặp nhầm lẫn lớn nhất: Lớp 0 (Spruce/Fir) và Lớp 1 (Lodgepole Pine) chiếm 70.88% tổng lỗi toàn mạng do chia sẻ cùng vành đai độ cao cận núi cao (2.700m - 3.200m) và thổ nhưỡng tương đồng.
3. Đột phá ở lớp thiểu số: Lớp 3 (chỉ 549 mẫu) đạt Recall 87.80% và F1 = 0.8419, tăng vượt trội so với Baseline (0.7728).

In [9]:
# 3. Điền kết quả vào bảng Excel experiments.xlsx
all_res = load_results(str(OUT_DIR / "results"))
rows = []
for r in all_res:
    if "cfg" in r and "summary" in r:
        exp_id = r["cfg"]["exp_id"]
        eval_s = None
        if exp_id == "cand-mwide-adam":
            eval_s = {"acc": 0.9268, "macro_f1": 0.8948}
        elif exp_id == "base-s1":
            eval_s = {"acc": 0.8875, "macro_f1": 0.8404}
        rows.append(to_row(r, eval_scores=eval_s))

template_path = REPO_ROOT / "templates" / "experiment_table_template.xlsx"
xlsx_out = OUT_DIR / "experiments.xlsx"
if template_path.exists() and len(rows) > 0:
    write_xlsx(rows, str(template_path), str(xlsx_out))
    print(f"File experiments.xlsx đã được cập nhật thành công: {xlsx_out}")


Đã lưu bảng kết quả thực nghiệm (24 dòng) -> D:\K4-Track4-Day1-Neuralnetwork\experiments.xlsx
File experiments.xlsx đã được cập nhật thành công: D:\K4-Track4-Day1-Neuralnetwork\experiments.xlsx
